# 第 2 章：數字與字串

> 🎬 **情境**
> 阿宏皺眉：「你印出來這樣我看不懂。給我一張**收據** —— 店名、品項、數量、單價、小計，還有含 5% 稅的總額。
> 會員 9 折的金額要算到**整數**，不能出現 58.5 元這種東西，我找不開。」

這章解決兩件事：**算得對**（數字）與**看得懂**（字串）。

## 2.1 😩 土法煉鋼：用加號黏字串

In [ ]:
品項 = "珍珠奶茶"
數量 = 3
單價 = 65
小計 = 單價 * 數量

print("品項:" + 品項 + " 數量:" + str(數量) + " 小計:" + str(小計) + "元")

能動，但：

1. 每個數字都要手動 `str()`，忘記一個就 `TypeError`
2. 加號和引號多到數不清
3. 看不出最後長什麼樣
4. 對齊？別想了

In [ ]:
# 忘記 str() 的下場
try:
    print("小計:" + 小計)
except TypeError as 錯誤:
    print("🔴 TypeError：", 錯誤)

## 2.2 🧪 先寫測試：順便把需求問清楚

阿宏說「9 折算到整數」。**58.5 要變成 58 還是 59？** 204.75 呢？

這種邊界問題如果不先寫測試，通常要等到阿宏發現少收錢才會爆出來。
我們去問了阿宏，他說「都給我四捨五入」。需求明確了，寫成測試：

In [ ]:
# 🔴 這些東西還不存在，先讓它紅一次
try:
    assert 含稅(195) == 205
    assert 會員價(65) == 59
except NameError as 錯誤:
    print("🔴 紅燈：", 錯誤)

> 🔍 **TDD 的第一個好處不是品質，是「逼你把模糊的需求問清楚」。**
> 寫不出 `assert`，代表你其實還不知道自己要做什麼。

## 2.3 💡 數字運算

In [ ]:
print("7 + 3  =", 7 + 3)
print("7 - 3  =", 7 - 3)
print("7 * 3  =", 7 * 3)
print("7 / 3  =", 7 / 3,  "← 除法結果永遠是 float")
print("7 // 3 =", 7 // 3, "← 整除，砍掉小數")
print("7 %  3 =", 7 % 3,  "← 餘數")
print("7 ** 3 =", 7 ** 3, "← 次方")

In [ ]:
# ⚠️ / 一定給你 float，即使除得盡
print(10 / 2, type(10 / 2))
print(10 // 2, type(10 // 2))

### `//` 和 `%` 的真實用途

阿宏問：「196 杯要幾個提袋？一袋裝 6 杯。」

In [ ]:
杯數 = 196
一袋裝 = 6

整袋 = 杯數 // 一袋裝
剩下 = 杯數 % 一袋裝
提袋數 = 整袋 + (1 if 剩下 else 0)      # 最後那幾杯也要一個袋子

print(f"{杯數} 杯 → 滿袋 {整袋} 個，剩 {剩下} 杯，共需 {提袋數} 個提袋")
assert 提袋數 == 33
print("🟢 通過")

In [ ]:
# % 最常見的另一個用途：判斷整除
for n in [10, 15, 100, 101]:
    print(f"{n:>4} 是偶數？{n % 2 == 0:<6} 是 100 的倍數？{n % 100 == 0}")

### ⚠️ round() 的銀行家捨入

Python 的 `round()` 遇到「正好 .5」時會進位到最接近的**偶數**。
這不是 bug，是 IEEE 754 國際標準，目的是讓大量數據平均後不會系統性偏高 ——
但它跟台灣人學的「逢五進一」不一樣，阿宏會覺得你算錯。

In [ ]:
for n in [0.5, 1.5, 2.5, 3.5, 58.5]:
    print(f"round({n}) = {round(n)}")

要「逢五一律進位」，用標準庫的 `decimal`：

In [ ]:
from decimal import Decimal, ROUND_HALF_UP


def 四捨五入(數值) -> int:
    """台灣人習慣的四捨五入（逢五進一）。"""
    return int(Decimal(str(數值)).quantize(Decimal("1"), rounding=ROUND_HALF_UP))


for n in [0.5, 2.5, 58.5, 204.75]:
    print(f"四捨五入({n}) = {四捨五入(n)}")

assert 四捨五入(58.5) == 59
assert 四捨五入(2.5) == 3
print("🟢 符合阿宏的直覺了")

### ⚠️ 浮點數的經典陷阱

In [ ]:
print("0.1 + 0.2 =", 0.1 + 0.2)
print("0.1 + 0.2 == 0.3 ?", 0.1 + 0.2 == 0.3)

這不是 Python 的 bug，所有用二進位表示小數的語言都一樣（JavaScript、Java、C…）。
就像十進位寫不出 1/3（0.333…無限循環），二進位也寫不出 0.1。

| 場景 | 做法 |
| --- | --- |
| 比較浮點數 | 用 `math.isclose(a, b)`，不要用 `==` |
| **金額計算** | **用整數存「元」或「分」，或用 `decimal.Decimal`** |
| 顯示 | f-string 格式化 `f"{x:.2f}"` |

In [ ]:
import math

print(math.isclose(0.1 + 0.2, 0.3))
assert math.isclose(0.1 + 0.2, 0.3)
print("🟢 這樣比較才安全")

## 2.4 💡 f-string：把值嵌進文字

記一件事就好：**引號前面加 `f`，用 `{}` 包住變數。**

In [ ]:
品項 = "珍珠奶茶"
數量 = 3
單價 = 65

print(f"{品項} x{數量} = {單價 * 數量} 元")

`{}` 裡面可以放**任何 Python 運算式**，而且**自動轉字串**，不用 `str()`。

In [ ]:
print("舊寫法：", "品項:" + 品項 + " 小計:" + str(單價 * 數量) + "元")
print("f-string：", f"品項:{品項} 小計:{單價 * 數量}元")

### 格式化語法：`{值:格式}`

In [ ]:
print(f"小數兩位  {3.14159:.2f}")
print(f"千分位    {1234567:,}")
print(f"百分比    {0.9:.0%}")
print(f"靠左寬10 [{'珍奶':<10}]")
print(f"靠右寬10 [{'珍奶':>10}]")
print(f"置中寬10 [{'珍奶':^10}]")
print(f"補零     {7:03d}")

In [ ]:
# 用它排出一張對得齊的表
print(f"{'品項':<12}{'數量':>4}{'小計':>8}")
print(f"{'珍珠奶茶':<12}{3:>4}{195:>8}")
print(f"{'冬瓜檸檬':<12}{2:>4}{90:>8}")

> ⚠️ **中文對齊會歪**：一個中文字在終端機佔兩格寬，但 Python 算它 1 個字元。
> 小專案忍耐一下；要完美對齊得用 `wcwidth` 套件。

### `{變數=}` 除錯神器（Python 3.8+）

In [ ]:
單價 = 65
數量 = 3
print(f"{單價=}, {數量=}, {單價 * 數量=}")

不用再寫 `print("單價", 單價)`，Python 幫你連名字一起印。**強烈建議記住這招。**

## 2.5 字串的常用操作

阿宏的 LINE 訂單長這樣（真實世界的資料永遠是髒的）：

In [ ]:
原始 = "  珍珠奶茶 , 大杯 , 65  \n"
print(repr(原始))          # repr() 會把空白和 \n 顯示出來，除錯時很好用

In [ ]:
print(repr(原始.strip()))            # 去頭尾空白與換行
print(原始.replace(" ", ""))         # 去掉所有空白
print(原始.split(","))               # 用逗號切開 → 清單
print("珍珠" in 原始)                 # 包含判斷（最常用）
print("珍珠奶茶".startswith("珍珠"))
print(len("珍珠奶茶"))
print("-" * 30)                      # 字串可以乘以數字！
print("、".join(["珍奶", "紅茶", "冬瓜"]))

In [ ]:
# 實戰：把髒資料清成乾淨的三個值
欄位 = [x.strip() for x in 原始.strip().split(",")]   # 串列推導式，第 5 章會細講
品項, 杯型, 價格 = 欄位[0], 欄位[1], int(欄位[2])

print(f"{品項=} {杯型=} {價格=}")
assert 品項 == "珍珠奶茶"
assert 價格 == 65 and type(價格) is int
print("🟢 清洗成功")

### ⚠️ 字串是「不可變」的

In [ ]:
名稱 = "珍珠奶茶"
名稱.replace("珍珠", "椰果")
print("沒接住結果：", 名稱)        # 還是珍珠奶茶！

名稱 = 名稱.replace("珍珠", "椰果")
print("接住結果：", 名稱)

`.replace()` 不會改動原字串，它**回傳一個新的**。

> 🔍 這個性質叫 **immutable（不可變）**。`str`、`int`、`float`、`bool`、`tuple` 都是不可變的。
> 第 5 章會遇到可變的 `list` 和 `dict`，到時這個差別會變得非常重要。

### 多行字串與跳脫字元

In [ ]:
print("第一行\n第二行")
print("欄位\t欄位")
print("他說:\"好喝\"")
print(r"C:\Users\新增")      # r 開頭 = 原始字串，\n 不會被當成換行

In [ ]:
# ⚠️ Windows 路徑的經典災難：\n 被當成換行、\t 被當成 Tab
災難 = "C:\temp\new.txt"
print("錯誤示範（字串裡其實藏了 Tab 和換行）：")
print(repr(災難))
print(災難)

print("\n正確做法 1（原始字串）：", repr(r"C:\temp\new.txt"))
print("正確做法 2（用斜線，Windows 也吃）：", "C:/temp/new.txt")
print("最佳做法：第 9 章的 pathlib，從此不用自己處理斜線")

## 2.6 🟢 讓測試變綠：完成收據

In [ ]:
稅率 = 0.05
會員折扣 = 0.9


def 含稅(金額: int) -> int:
    return 四捨五入(金額 * (1 + 稅率))


def 會員價(金額: int) -> int:
    return 四捨五入(金額 * 會員折扣)


品項, 數量, 單價 = "珍珠奶茶", 3, 65
小計 = 單價 * 數量

收據 = f"""{'阿宏手搖飲':^20}
{'-' * 24}
{'品項':<10}{'數量':>4}{'小計':>8}
{品項:<10}{數量:>4}{小計:>8}
{'-' * 24}
{'含稅總計':<14}{含稅(小計):>8}
"""
print(收據)

In [ ]:
# 🟢 第 2.2 節那些紅燈，現在應該全綠
assert 小計 == 195
assert 含稅(195) == 205
assert 會員價(65) == 59
assert "珍珠奶茶" in 收據
print("🟢 全部通過")

## 📌 本章速記

In [ ]:
# 數字
assert 7 / 3 != 2                  # 除法永遠是 float
assert 7 // 3 == 2                 # 整除
assert 7 % 3 == 1                  # 餘數
assert 7 ** 3 == 343               # 次方
assert round(2.5) == 2             # ⚠️ 銀行家捨入
assert 0.1 + 0.2 != 0.3            # ⚠️ 浮點數
assert math.isclose(0.1 + 0.2, 0.3)

# f-string
assert f"{3.14159:.2f}" == "3.14"
assert f"{1234567:,}" == "1,234,567"
assert f"{0.9:.0%}" == "90%"

# 字串
assert "  a ".strip() == "a"
assert "a,b".split(",") == ["a", "b"]
assert "、".join(["a", "b"]) == "a、b"
assert "-" * 3 == "---"
print("🟢 速記全部通過")

---
## 🧪 練習

### 練習 2-1：計算找零
客人點 3 杯珍奶（65）、2 杯冬瓜檸檬（45），給了 500 元。

In [ ]:
# 👉 請算出 總額 與 找零


try:
    assert 總額 == 285
    assert 找零 == 215
    print("🟢 2-1 通過")
except NameError as 錯誤:
    print("🔴 還沒做：", 錯誤)

### 練習 2-2：提袋數量
一袋裝 8 杯，請用 `//` 和 `%` 算出 197 杯要幾個袋子（裝不滿的也算一袋）。

In [ ]:
# 👉 請算出 提袋數答案


try:
    assert 提袋數答案 == 25
    print("🟢 2-2 通過")
except NameError as 錯誤:
    print("🔴 還沒做：", 錯誤)

### 練習 2-3：清洗 LINE 訂單

In [ ]:
髒資料 = "  冬瓜檸檬 | 中杯 | 45  \n"

# 👉 請處理出 乾淨品項（str）與 乾淨價格（int）


try:
    assert 乾淨品項 == "冬瓜檸檬"
    assert 乾淨價格 == 45
    assert type(乾淨價格) is int
    print("🟢 2-3 通過")
except NameError as 錯誤:
    print("🔴 還沒做：", 錯誤)

### 練習 2-4：對齊的一行

用**一個** f-string 產生報表的一行：
品項靠左寬 12、數量靠右寬 6 且千分位、金額靠右寬 10 且千分位。

In [ ]:
報表品項, 報表數量, 報表金額 = "珍珠奶茶", 1234, 80210

行 = None  # 👉 改成你的 f-string

try:
    assert isinstance(行, str), "請寫成一個 f-string"
    assert len(行) == 28, f"總寬度應該是 12+6+10=28，你的是 {len(行)}"
    assert 行.startswith("珍珠奶茶    "), "品項要靠左"
    assert 行.endswith("    80,210"), "金額要靠右且有千分位"
    assert " 1,234" in 行, "數量要有千分位"
    print("🟢 2-4 通過")
except AssertionError as 錯誤:
    print("🔴 還沒做或不正確：", 錯誤)

---
➡️ 下一章：`03-判斷.ipynb` —— 阿宏要搞會員折扣了，價格不再是固定的。